In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cars.csv")
print("Start:", df.shape)

Start: (762091, 20)


In [20]:
df.describe()

,year,mileage,accidents_or_damage,one_owner,personal_use_only,seller_rating,driver_rating,driver_reviews_num,price_drop,price
count,762091.000000,7.615850e+05,737879.000000,730608.000000,737239.000000,548118.000000,730459.000000,762091.000000,410112.000000,7.620910e+05
mean,2017.791398,5.578169e+04,0.228616,0.561969,0.657212,4.158568,4.623523,89.550900,1007.467068,3.648898e+04
std,5.110532,4.355788e+04,0.419942,0.496145,0.474642,0.805741,0.276902,115.082266,1375.122208,1.984183e+06
min,1915.000000,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,100.000000,1.000000e+00
25%,2016.000000,2.328700e+04,0.000000,0.000000,0.000000,3.800000,4.500000,14.000000,380.000000,1.958300e+04
50%,2019.000000,4.559600e+04,0.000000,1.000000,1.000000,4.500000,4.700000,51.000000,642.000000,2.798900e+04
75%,2021.000000,7.836500e+04,0.000000,1.000000,1.000000,4.700000,4.800000,119.000000,1007.000000,3.948800e+04
max,2024.000000,1.119067e+06,1.000000,1.000000,1.000000,5.000000,5.000000,1025.000000,170995.000000,1.000000e+09


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 762091 entries, 0 to 762090
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   manufacturer         762091 non-null  str    
 1   model                762091 non-null  str    
 2   year                 762091 non-null  int64  
 3   mileage              761585 non-null  float64
 4   engine               747041 non-null  str    
 5   transmission         752187 non-null  str    
 6   drivetrain           740529 non-null  str    
 7   fuel_type            739164 non-null  str    
 8   mpg                  620020 non-null  str    
 9   exterior_color       753232 non-null  str    
 10  interior_color       705116 non-null  str    
 11  accidents_or_damage  737879 non-null  float64
 12  one_owner            730608 non-null  float64
 13  personal_use_only    737239 non-null  float64
 14  seller_name          753498 non-null  str    
 15  seller_rating        548118 

In [22]:
#dropping the missing mileage rows
df = df.dropna(subset=["mileage"])
print(df.shape)

(761585, 20)


#### Dealing with price only including values between 2k - 200k

In [23]:
#dropping rows for price outside of range 2k - 200k
df=df[df["price"].between(2000, 200000)]

In [24]:
df["price_drop"] = df["price_drop"].fillna(0)

#### Dealing with seller rating, adding flag variable

In [25]:
g = df.groupby("seller_name")["seller_rating"]
mixed = g.apply(lambda s: s.isna().any() and s.notna().any())
print("Sellers with some ratings missing and some present:", mixed.sum())

Sellers with some ratings missing and some present: 337


In [26]:
df["seller_rating"] = df["seller_rating"].fillna(
    df.groupby("seller_name")["seller_rating"].transform("median")
)

In [27]:
df["seller_rating_missing"] = df["seller_rating"].isna().astype(int)

#### Dealing with MPG grouping cars that have same make/model/year and giving them the same mpg

In [28]:
df[["year","manufacturer", "model","mpg"]]

,year,manufacturer,model,mpg
0,2013,Acura,ILX Hybrid 1.5L,39-38
1,2013,Acura,ILX Hybrid 1.5L,39-38
2,2013,Acura,ILX Hybrid 1.5L,39-38
3,2013,Acura,ILX Hybrid 1.5L,39-38
4,2013,Acura,ILX Hybrid 1.5L,39-38
...,...,...,...,...
762086,2022,Volvo,S60 B5 Momentum,25-33
762087,2012,Volvo,S60 T5,NaN
762088,2014,Volvo,S60 T5,21-30
762089,2013,Volvo,S60 T5 Platinum,20-29


In [29]:
mpg_split = df["mpg"].str.split("-", expand=True)
df["mpg_city"] = pd.to_numeric(mpg_split[0], errors="coerce")
df["mpg_hwy"] = pd.to_numeric(mpg_split[1], errors = 'coerce')

In [30]:
df[["mpg_city","mpg_hwy"]].describe()

,mpg_city,mpg_hwy
count,618728.000000,615877.000000
mean,21.316918,27.953605
std,6.500205,6.937287
min,0.000000,0.000000
25%,17.000000,24.000000
50%,20.000000,28.000000
75%,24.000000,32.000000
max,126.000000,255.000000


In [31]:
for col in ["mpg_city", "mpg_hwy"]:
    before =df[col].isna().sum()

    df[col] = df[col].fillna(df.groupby(["manufacturer","year","model"])[col].transform("median"))

    df[col] = df[col].fillna(df.groupby(["manufacturer", "model"])[col].transform("median"))

    print(f"{col}: filled {before - df[col].isna().sum():,}, still missing {df[col].isna().sum():,}")

mpg_city: filled 109,809, still missing 31,443
mpg_hwy: filled 112,400, still missing 31,703


In [34]:
df["fuel_type"].value_counts()

fuel_type
Gasoline          643008
Hybrid             30625
Diesel             27907
Unknown            22861
Flex Fuel          19243
Electric           16176
Plug-In Hybrid       121
Other                 39
Name: count, dtype: int64

In [ ]:
fuel_groups = {
    "Gasoline": ["Gasoline", "Gasoline Fuel", "Regular Unleaded", "Premium Unleaded",
                 "Premium", "Premium (Required)", "Gas", "G"],
    "Flex Fuel": ["E85 Flex Fuel", "Flexible Fuel", "Flex Fuel Capability",
                  "Flex Fuel", "E85 Fl", "Bi-Fuel"],
    "Diesel": ["Diesel", "Diesel Fuel", "Biodiesel", "Bio Diesel"],
    "Hybrid": ["Hybrid", "Hybrid Fuel", "Gasoline/Mild Electric Hybrid",
               "Gas/Electric Hybrid", "B"],
    "Plug-In Hybrid": ["Plug-In Hybrid", "PHEV", "Plug-In Electric/Gas"],
    "Electric": ["Electric", "Electric Fuel System"],
    "Other": ["Compressed Natural Gas", "Natural Gas", "Gaseous",
              "Hydrogen Fuel Cell", "Other"],
}

fuel_map = {variant: clean for clean, variants in fuel_groups.items() for variant in variants}

df["fuel_type"] = df["fuel_type"].map(fuel_map).fillna("Unknown")
df["fuel_type"].value_counts()

fuel_type
Gasoline          643008
Hybrid             30625
Diesel             27907
Unknown            22861
Flex Fuel          19243
Electric           16176
Plug-In Hybrid       121
Other                 39
Name: count, dtype: int64